# core.dom.embedding.object

> Recursive summary embeddings with raw evidence and an object relationship catalog.

Index nonempty object `summary` fields and the document summary in `*_semantics.json`.
Chroma holds summary vectors; SQLite holds exact source content, stable logical IDs,
source IDs, canonical/alternate paths, ordered parent/child relationships, and section context.
Section/prose `s` fields are context rather than additional vector records.

Copies in DOM wrappers, table headers, cells and subnodes become one logical record.
An object's source ID is scoped to its normalized semantics-file path. Moving a source
creates a different document identity. Repeated equal summaries remain separate objects.

For corpus configuration, manual indexing, live concurrency status, and evidence
retrieval, use [the object embedding application](05.application.embedding.object.ipynb).


In [ ]:
# | default_exp core.dom.embedding.object

## Canonical records

`collect_object_records` is an offline, non-mutating traversal. Paths begin at `$` and
escape `/` and `~` in keys. Owning `children` records precede `cells`, which precede
header/wrapper mirrors. Conflicting type/content values for the same ID fail visibly.
The preferred summary wins; empty summaries stay in the catalog without a vector.
The document retains the original JSON text; object `content` retains its exact source
markup. `attributes` preserves captions, line ranges, image links, and list/table details.


In [ ]:
# | export
from __future__ import annotations

import asyncio
import hashlib
import json
import math
import sqlite3
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any
from uuid import uuid4

from ribosome.core.dom.embedding.natural_text import get_embeddings_with_retry
from ribosome.core.document import Document, DocumentGraph, Node, SourceSpan, SummaryArtifact, source_revision, referenced_asset_fingerprints
from ribosome.core.artifacts import ArtifactRepository
from ribosome.retrieval.contracts import EncoderSpec, RepresentationSpec, IndexManifest, lexical_terms, token_count, atomic_json, index_lock, protected_generations, verify_encoder, pin_generations


def _json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))


def _digest(value):
    return hashlib.sha256(_json(value).encode('utf-8')).hexdigest()


def _pointer(path, key):
    return path + '/' + str(key).replace('~', '~0').replace('/', '~1')


def _heading_text(value):
    if isinstance(value, dict):
        if value.get('t') == 'Str':
            return str(value.get('c', ''))
        if value.get('t') in {'Space', 'SoftBreak', 'LineBreak'}:
            return ' '
        return _heading_text(value.get('c'))
    if isinstance(value, list):
        return ''.join(_heading_text(item) for item in value)
    return ''


@dataclass
class ObjectRecord:
    """A logical object; IDs reference other catalog records, not Chroma rows."""
    id: str
    document_id: str
    source_id: str | int | None
    type: str
    object_path: str
    occurrence_paths: list[str]
    semantics_file: str
    source_file: str
    summary: str
    content: str
    content_format: str
    parent_id: str | None
    children_ids: list[str] = field(default_factory=list)
    section_headings: list[str] = field(default_factory=list)
    attributes: dict[str, Any] = field(default_factory=dict)


@dataclass
class IndexingResult:
    semantics_file: str
    status: str
    document_id: str = ''
    generation_id: str = ''
    objects: int = 0
    embedded: int = 0
    reused: int = 0
    skipped: int = 0
    error: str = ''


@dataclass
class ObjectSearchHit:
    distance: float
    record: ObjectRecord
    ancestors: list[ObjectRecord]
    children: list[ObjectRecord]

    @property
    def summary(self):
        return self.record.summary


def collect_object_records(tree: dict, semantics_file: str | Path, *, raw_json: str | None = None, include_semantic_nodes: bool = False) -> list[ObjectRecord]:
    """Collect one record per typed source ID without changing the input tree.

    children precede cells; wrapper subnodes and headers are mirrors. Structural
    containment determines relationships. Paths start at $ and escape / and ~.
    """
    source = str(Path(semantics_file).resolve())
    if not isinstance(tree, dict):
        raise ValueError(f'{source}: semantic JSON must be an object')
    document_id = _digest(['document', source])
    occurrences = {}

    def scan(node, path='$', parent=None, headings=(), penalty=0):
        if isinstance(node, list):
            for i, child in enumerate(node):
                scan(child, _pointer(path, i), parent, headings, penalty)
            return
        if not isinstance(node, dict):
            return
        if node.get('t') == 'Section':
            content = node.get('c', [])
            header = content[0] if isinstance(content, list) and content else {}
            title = _heading_text(header).strip()
            if title:
                headings = (*headings, title)
        if 'id' in node and isinstance(node.get('type'), str) and ('summary' in node or 'content' in node):
            identifier = node['id']
            if isinstance(identifier, bool) or not isinstance(identifier, (str, int)):
                raise ValueError(f'{source} at {path}: object id must be a string or integer')
            key = _json(identifier)
            occurrences.setdefault(key, []).append((penalty, path, parent, headings, node))
            parent = key
        # Fixed priority is independent of source dictionary insertion order.
        keys = sorted(node, key=lambda k: (0 if k == 'children' else 1 if k == 'cells' else 2, k))
        for key in keys:
            if key in {'summary', 's'}:
                continue
            extra = 1 if key == 'cells' else 2 if key == 'headers' else 0
            if key == 'subnodes' and isinstance(node.get('object'), dict):
                extra += 10
            scan(node[key], _pointer(path, key), parent, headings, penalty + extra)

    scan(tree)
    selected = {key: min(items, key=lambda item: item[0]) for key, items in occurrences.items()}
    ids = {key: _digest(['object', document_id, key]) for key in selected}

    def summary(value, path):
        if not isinstance(value, str):
            raise ValueError(f'{source} at {path}: summary must be a string')
        return value

    document = ObjectRecord(
        id=document_id, document_id=document_id, source_id=None, type='document',
        object_path='$', occurrence_paths=['$'], semantics_file=source,
        source_file=str(tree.get('file_path') or ''), summary=summary(tree.get('summary', ''), '$'),
        content=raw_json if raw_json is not None else _json(tree), content_format='json',
        parent_id=None, attributes={'title': tree.get('title', '')})
    records = [document]
    for key, (_, path, parent, headings, node) in selected.items():
        for _, other_path, _, _, other in occurrences[key]:
            if other['type'] != node['type'] or other.get('content', '') != node.get('content', ''):
                raise ValueError(f'{source}: conflicting object {node["id"]!r} at {path} and {other_path}')
        content = node.get('content', '')
        if not isinstance(content, str):
            raise ValueError(f'{source} at {path}: object content must be a string')
        records.append(ObjectRecord(
            id=ids[key], document_id=document_id, source_id=node['id'], type='table' if node['type'] == 'task' else node['type'],
            object_path=path, occurrence_paths=list(dict.fromkeys(item[1] for item in occurrences[key])),
            semantics_file=source, source_file=str(node.get('source_file') or tree.get('file_path') or ''),
            summary=summary(node.get('summary', ''), path), content=content, content_format='source',
            parent_id=ids[parent] if parent is not None else document_id,
            section_headings=list(headings),
            attributes={**{k: v for k, v in node.items() if k not in {'children', 'cells', 'summary', 'content'}},
                **({'semantic_roles': sorted(set(node.get('semantic_roles', [])) | {'task'})} if node['type'] == 'task' else {})}))
    if include_semantic_nodes:
        def source_text(node):
            if isinstance(node, str): return node
            if isinstance(node, list): return ''.join(source_text(value) for value in node)
            if not isinstance(node, dict): return ''
            if node.get('t') in {'Space', 'SoftBreak', 'LineBreak'}: return ' '
            if node.get('t') in {'CodeBlock', 'Code', 'Math'}:
                return str(node.get('c', ['', ''])[-1])
            return source_text(node.get('c', ''))

        kinds = {'Section': 'section', 'Para': 'paragraph', 'Plain': 'paragraph',
                 'CodeBlock': 'code', 'Math': 'formula'}
        sections = []
        def semantic_scan(node, path='$', parent=document_id, headings=()):
            if isinstance(node, list):
                for i, value in enumerate(node): semantic_scan(value, _pointer(path, i), parent, headings)
                return
            if not isinstance(node, dict) or 'object' in node: return
            kind = kinds.get(node.get('t'))
            if kind:
                if kind == 'section':
                    title = _heading_text(node.get('c', [{}])[0]).strip()
                    if title: headings = (*headings, title)
                identifier = _digest(['semantic', document_id, path])
                content = source_text(node)
                records.append(ObjectRecord(identifier, document_id, path, kind, path, [path], source,
                    str(tree.get('file_path') or source), summary(node.get('s', ''), path), content, 'source',
                    parent, section_headings=list(headings), attributes={'source_locator': path, 'span_status': 'unknown'}))
                if kind == 'section': sections.append((path, identifier))
                parent = identifier
                # Inline descendants of a paragraph are source content, not duplicate evidence nodes.
                if kind in {'paragraph', 'code', 'formula'}: return
            for key in ('blocks', 'c', 'subnodes'):
                if key in node: semantic_scan(node[key], _pointer(path, key), parent, headings)
        semantic_scan(tree)
        for record in records[1:]:
            if record.parent_id == document_id:
                owners = [(path, identifier) for path, identifier in sections
                          if record.object_path.startswith(path + '/') and identifier != record.id]
                if owners: record.parent_id = max(owners, key=lambda item: len(item[0]))[1]
    by_id = {record.id: record for record in records}
    for record in records[1:]:
        if record.parent_id == record.id:
            raise ValueError(f'{source} at {record.object_path}: object contains itself')
        by_id[record.parent_id].children_ids.append(record.id)
        seen = {record.id}
        parent = record.parent_id
        while parent is not None:
            if parent in seen:
                raise ValueError(f'{source}: cyclic object relationships at {record.object_path}')
            seen.add(parent)
            parent = by_id[parent].parent_id
    # Rows may mirror only rich cells in children, while cells contains every
    # column. Relationships follow column order rather than mirror discovery.
    for record in records:
        if record.type == 'table_row':
            record.children_ids.sort(key=lambda identifier: (
                by_id[identifier].attributes.get('column', float('inf'))
                if isinstance(by_id[identifier].attributes.get('column'), int)
                else float('inf')))
    return records


## Persistent index and context search

`ObjectEmbeddingIndex` accepts injected clients/collections for testing. Its defaults
are a local Ollama client with `qwen3-embedding:8b`, a dedicated cosine `object_summaries` collection,
and `db/object_embeddings/qwen3-embedding-8b/catalog.sqlite3`. `embedding_config` identifies provider or
embedding-setting differences (for example `{"provider": "http://localhost:11434"}`);
use a new index when that configuration or the model changes. The client must be
configured to match this identity. Never mix object and natural-text vector collections.

Each successful file publishes a new immutable generation through a SQLite transaction.
Search only considers active generations and resolves context from the same generation.
Provider/storage failure or cancellation leaves the previous generation searchable.
Computed vectors are cached by summary and configuration for retries and metadata edits.
Unchanged files require no embedding calls. Inactive generations and interrupted Chroma
rows are retained for later maintenance. Source files and summary checkpoints are untouched.

`embedding_concurrency` bounds concurrent unique-summary requests within one document (default 1).
The application notebook sets it to the combined server capacity. Vectors are cached as
requests finish, then a single writer publishes the complete document. Failed or
cancelled preparation cancels and awaits all remaining workers.

Use one indexing writer per catalog; calls on one instance are serialized. `index_file`
raises errors; `index_paths` reports per-file failures and continues, but propagates
cancellation. Missing input roots fail visibly. `get_object` also returns objects with
empty summaries. `search` returns cosine distance, full records, root-first ancestors,
and immediate children, with optional document/type filters. Distances are not probabilities.
Call `close()` or use the index as a context manager to release SQLite resources.


In [ ]:
# | export
class ObjectEmbeddingIndex:
    """Summary vectors plus a versioned SQLite object catalog.

    One writer per index is supported. Queries resolve an immutable active
    generation snapshot, including context. Inactive generations are retained.
    Changing model, provider identity/configuration or collection requires a new
    index. Input semantics and summary checkpoints are never written.
    """

    def __init__(self, catalog_path: str | Path = 'db/object_embeddings/qwen3-embedding-8b/catalog.sqlite3', *,
                 collection=None, llm_client=None, model: str = 'qwen3-embedding:8b',
                 embedding_config: dict | None = None, embedding_concurrency: int = 1,
                 encoder_spec: EncoderSpec | None = None, representations=('summary',), write_batch_size=256,
                 include_semantic_nodes=False, artifact_repository=None):
        self.catalog_path = Path(catalog_path).resolve()
        self.catalog_path.parent.mkdir(parents=True, exist_ok=True)
        if not isinstance(model, str) or not model.strip():
            raise ValueError('model must be a nonempty string')
        if isinstance(embedding_concurrency, bool) or not isinstance(embedding_concurrency, int) or embedding_concurrency < 1:
            raise ValueError('embedding_concurrency must be a positive integer')
        self.embedding_concurrency = embedding_concurrency
        self.model = model
        self.embedding_config = dict(embedding_config or {})
        self.encoder_spec = encoder_spec
        self._encoder_preflight_done = False
        self.include_semantic_nodes = include_semantic_nodes or 'source' in representations
        if include_semantic_nodes: self.embedding_config['include_semantic_nodes'] = True
        self.artifact_repository = artifact_repository or ArtifactRepository(self.catalog_path.parent / 'artifacts')
        self.representations = tuple(dict.fromkeys(representations))
        if not self.representations or not set(self.representations) <= {'summary', 'source'}:
            raise ValueError('Representations must contain summary and/or source')
        if 'source' in self.representations and encoder_spec is None:
            raise ValueError('Source views require an explicit verified EncoderSpec')
        if encoder_spec is not None:
            if encoder_spec.model != model: raise ValueError('Encoder model does not match index model')
            self.embedding_config.update(encoder_spec=asdict(encoder_spec), representations=list(self.representations))
        if isinstance(write_batch_size, bool) or write_batch_size < 1:
            raise ValueError('write_batch_size must be positive')
        self.write_batch_size = write_batch_size
        self.fingerprint = _digest({'schema': 1, 'model': model, 'config': self.embedding_config})
        if llm_client is None:
            from ollama import AsyncClient
            llm_client = AsyncClient()
        self.llm_client = llm_client
        if collection is None:
            from chromadb import PersistentClient
            self.db_client = PersistentClient(path=str(self.catalog_path.parent / 'chroma'))
            collection = self.db_client.get_or_create_collection(
                'object_summaries', metadata={'hnsw:space': 'cosine'}, embedding_function=None)
        self.collection = collection
        metadata = dict(collection.metadata or {})
        configuration = getattr(collection, 'configuration', {}) or {}
        metric = metadata.get('hnsw:space') or (configuration.get('hnsw') or {}).get('space')
        if metric != 'cosine':
            raise ValueError('Object embeddings require a dedicated cosine Chroma collection')
        self.connection = sqlite3.connect(self.catalog_path)
        self.connection.row_factory = sqlite3.Row
        self._writer = asyncio.Lock()
        try:
            self.connection.executescript('''
                CREATE TABLE IF NOT EXISTS settings (key TEXT PRIMARY KEY, value TEXT NOT NULL);
                CREATE TABLE IF NOT EXISTS generations (
                    id TEXT PRIMARY KEY, document_id TEXT NOT NULL, source TEXT NOT NULL,
                    signature TEXT NOT NULL, embedded_count INTEGER NOT NULL);
                CREATE TABLE IF NOT EXISTS active_documents (
                    document_id TEXT PRIMARY KEY, generation_id TEXT NOT NULL);
                CREATE TABLE IF NOT EXISTS objects (
                    generation_id TEXT NOT NULL, id TEXT NOT NULL, record TEXT NOT NULL,
                    vector_id TEXT, PRIMARY KEY (generation_id, id));
                CREATE TABLE IF NOT EXISTS vectors (
                    cache_key TEXT PRIMARY KEY, vector TEXT NOT NULL, actual_model TEXT NOT NULL);
                CREATE TABLE IF NOT EXISTS object_metadata (
                    generation_id TEXT NOT NULL, id TEXT NOT NULL, document_id TEXT NOT NULL,
                    parent_id TEXT, kind TEXT NOT NULL, vector_id TEXT, source_file TEXT NOT NULL,
                    semantics_file TEXT NOT NULL, PRIMARY KEY(generation_id,id));
                CREATE INDEX IF NOT EXISTS object_metadata_filter ON object_metadata(generation_id,kind,document_id);
                CREATE TABLE IF NOT EXISTS object_representations (
                    generation_id TEXT NOT NULL, object_id TEXT NOT NULL, kind TEXT NOT NULL,
                    part INTEGER NOT NULL, vector_id TEXT NOT NULL, input_hash TEXT NOT NULL,
                    source_map TEXT NOT NULL, PRIMARY KEY(generation_id,object_id,kind,part));
                CREATE INDEX IF NOT EXISTS object_representations_filter ON object_representations(generation_id,kind);
                CREATE VIRTUAL TABLE IF NOT EXISTS object_fts USING fts5(generation_id UNINDEXED, object_id UNINDEXED, tokens);
                CREATE TABLE IF NOT EXISTS object_terms (generation_id TEXT NOT NULL, term TEXT NOT NULL, object_id TEXT NOT NULL, PRIMARY KEY(generation_id,term,object_id));
                CREATE TABLE IF NOT EXISTS generation_pins (token TEXT PRIMARY KEY, generation_id TEXT NOT NULL, expires REAL NOT NULL);
                CREATE TABLE IF NOT EXISTS generation_lifecycle (generation_id TEXT PRIMARY KEY, created REAL NOT NULL);
                CREATE TABLE IF NOT EXISTS corpus_tombstones (document_id TEXT PRIMARY KEY, source TEXT NOT NULL, manifest_id TEXT NOT NULL);
                CREATE TABLE IF NOT EXISTS document_graphs (generation_id TEXT PRIMARY KEY, logical_document_id TEXT NOT NULL, revision_id TEXT NOT NULL, graph TEXT NOT NULL);
            ''')
            settings = dict(self.connection.execute('SELECT key, value FROM settings').fetchall())
            identity = {'fingerprint': self.fingerprint, 'collection_id': str(collection.id)}
            if settings and json.loads(settings['identity']) != identity:
                raise ValueError('Catalog embedding configuration or Chroma collection mismatch; use a new index')
            catalog_id = json.loads(settings['catalog_id']) if settings else uuid4().hex
            expected = {'object_index_config': self.fingerprint, 'object_catalog_id': catalog_id}
            if any(key in metadata and metadata[key] != value for key, value in expected.items()):
                raise ValueError('Chroma model/configuration or object catalog mismatch; use a new collection')
            if collection.count() and not all(metadata.get(key) == value for key, value in expected.items()):
                raise ValueError('Refusing to adopt a populated collection without matching object-index provenance')
            if not settings:
                with self.connection:
                    self.connection.executemany('INSERT INTO settings VALUES (?, ?)',
                        [('identity', _json(identity)), ('catalog_id', _json(catalog_id))])
            # Own only these metadata fields; retain Chroma's distance configuration.
            if any(metadata.get(key) != value for key, value in expected.items()):
                collection.modify(metadata={**{k: v for k, v in metadata.items() if not k.startswith('hnsw:')}, **expected})
            self.dimension = json.loads(settings['dimension']) if 'dimension' in settings else None
            if encoder_spec is not None and self.dimension is not None and self.dimension != encoder_spec.dimension:
                raise ValueError('Encoder dimension does not match catalog')
            self._backfill_metadata()
            self._write_manifest()
        except BaseException:
            self.connection.close()
            raise

    def _document_graph(self, records, tree, semantics_file):
        """Validate the unique catalog projection without reinterpreting stale legacy mirrors."""
        source = Path(tree.get('file_path') or str(semantics_file).replace('_semantics.json', '.md').replace('_tasks.json', '.md'))
        if not source.is_absolute(): source = source.resolve() if source.is_file() else semantics_file.parent / source
        document = self.artifact_repository.adopt_document(str(source.resolve()), title=str(tree.get('title', '')))
        available = source.is_file()
        observed_bytes = source.read_bytes() if available else None
        observed_assets = referenced_asset_fingerprints(source) if available else {}
        declared = tree.get('_source_provenance') or {}
        verified = available and bool(declared)
        if verified:
            if declared.get('source_sha256') != hashlib.sha256(observed_bytes).hexdigest() or declared.get('assets', {}) != observed_assets:
                raise ValueError('Semantic output source provenance is stale; publish a compatible summary artifact before indexing')
        content = observed_bytes if verified else _json([(record.id, record.content) for record in records[1:]]).encode()
        assets = observed_assets if verified else {'historical_identity': 'unknown', 'observed_assets': observed_assets}
        revision = source_revision(document.id, str(source), content, assets=assets,
            parser_manifest={'input_provenance': 'verified-source' if verified else 'legacy-source-projection',
                'observed_source_sha256': hashlib.sha256(observed_bytes).hexdigest() if available else None,
                'asset_provenance': declared.get('asset_provenance', 'historical_unknown'), 'coverage': 'unknown'})
        identities = {record.id: _digest([document.id, revision.id, record.id]) for record in records}
        nodes, spans, summaries = [], [], []
        for record in records:
            identifier = identities[record.id]
            record.attributes.update(source_revision_id=revision.id, logical_document_id=document.id,
                source_hash=revision.content_sha256, source_identity_status='observed' if verified else 'historical_unknown',
                observed_source_sha256=hashlib.sha256(observed_bytes).hexdigest() if available else None)
            attrs = dict(record.attributes)
            span_ids = []
            start, end = attrs.get('line_start'), attrs.get('line_end')
            if start is not None and end is not None:
                span_id = _digest([revision.id, record.source_file, start, end])
                if not any(span.id == span_id for span in spans):
                    spans.append(SourceSpan(span_id, revision.id, record.source_file or str(source), int(start), int(end)))
                span_ids.append(span_id)
            nodes.append(Node(identifier, revision.id, record.type, record.content,
                identities.get(record.parent_id), [identities[value] for value in record.children_ids],
                list(attrs.get('semantic_roles', [])), span_ids, attrs,
                [record.id] + (['source:' + _json(record.source_id)] if record.source_id is not None else [])))
            if record.summary:
                summaries.append(SummaryArtifact(identifier, revision.id, record.summary))
        return DocumentGraph(document, revision, nodes, spans=spans, summaries=summaries,
            provenance={'adapter': 'object-catalog-v1', 'legacy_document_id': records[0].id}).validate()

    def _backfill_metadata(self):
        """One-time additive metadata migration; no vectors or summaries are regenerated."""
        rows = self.connection.execute('SELECT o.generation_id,o.id,o.record,o.vector_id FROM objects o LEFT JOIN object_metadata m ON m.generation_id=o.generation_id AND m.id=o.id WHERE m.id IS NULL').fetchall()
        with self.connection:
            for generation, identifier, raw, vector in rows:
                record = json.loads(raw)
                self._insert_metadata(generation, record, vector)

    def _insert_metadata(self, generation, record, vector):
        kind = 'table' if record['type'] == 'task' else record['type']
        self.connection.execute('INSERT INTO object_metadata VALUES (?,?,?,?,?,?,?,?)',
            (generation, record['id'], record['document_id'], record.get('parent_id'), kind, vector,
             record.get('source_file', ''), record.get('semantics_file', '')))
        if kind != 'document':
            self.connection.execute('INSERT INTO object_fts VALUES (?,?,?)',
                (generation, record['id'], ' '.join(lexical_terms(record.get('content', '')))))
            self.connection.executemany('INSERT OR IGNORE INTO object_terms VALUES (?,?,?)', [(generation, term, record['id']) for term in lexical_terms(record.get('content', ''))])

    def _write_manifest(self):
        settings = dict(self.connection.execute('SELECT key,value FROM settings'))
        sources = [row[0] for row in self.connection.execute('SELECT source FROM generations g JOIN active_documents a ON a.generation_id=g.id')]
        kinds = {'tasks' if value.endswith('_tasks.json') else 'semantics' for value in sources}
        manifest = IndexManifest(next(iter(kinds)) if len(kinds) == 1 else 'mixed',
            getattr(self.collection, 'name', 'object_summaries'), json.loads(settings['catalog_id']), self.fingerprint,
            [row[0] for row in self.connection.execute('SELECT DISTINCT kind FROM object_metadata ORDER BY kind')],
            list(self.representations), dict(self.connection.execute('SELECT document_id,generation_id FROM active_documents')),
            embedding_config=self.embedding_config, encoder_spec=asdict(self.encoder_spec) if self.encoder_spec else None)
        manifest.write(self.catalog_path.parent)

    def close(self):
        self.connection.close()

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()

    def _validate_vector(self, vector):
        if not isinstance(vector, (list, tuple)) or not vector:
            raise ValueError('Embedding must be a nonempty vector')
        if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) for v in vector):
            raise ValueError('Embedding must contain finite numeric values')
        if not any(vector):
            raise ValueError('Cosine embedding must have nonzero length')
        if self.encoder_spec is not None and len(vector) != self.encoder_spec.dimension:
            raise ValueError('Embedding dimension does not match EncoderSpec')
        if self.dimension is not None and len(vector) != self.dimension:
            raise ValueError(f'Embedding dimension mismatch: expected {self.dimension}, got {len(vector)}')
        if self.dimension is None:
            with self.connection:
                self.connection.execute('INSERT OR IGNORE INTO settings VALUES (?, ?)', ('dimension', _json(len(vector))))
            self.dimension = json.loads(self.connection.execute(
                "SELECT value FROM settings WHERE key='dimension'").fetchone()[0])
            if len(vector) != self.dimension:
                raise ValueError('Embedding dimension mismatch')
        values = [float(v) for v in vector]
        if self.encoder_spec is not None and self.encoder_spec.normalization == 'l2':
            norm = math.sqrt(sum(value * value for value in values)); values = [value / norm for value in values]
        return values

    async def _embedding(self, text, label, *, cache=True, query=False):
        if self.encoder_spec is not None and not self._encoder_preflight_done:
            self._encoder_metadata_verified = await verify_encoder(self.llm_client, self.encoder_spec)
            self._encoder_preflight_done = True
        rendered = self.encoder_spec.render(text, query=query) if self.encoder_spec else text
        key = _digest([self.fingerprint, rendered])
        cached = self.connection.execute('SELECT vector, actual_model FROM vectors WHERE cache_key=?', (key,)).fetchone() if cache else None
        if cached is not None:
            return self._validate_vector(json.loads(cached['vector'])), cached['actual_model'], True
        if self.encoder_spec is None:
            vector, response = await get_embeddings_with_retry(text, label, self.llm_client, self.model)
        else:
            response = await self.llm_client.embed(**self.encoder_spec.request(text, query=query))
            vector = response['embeddings'][0]
            reported = response.get('model_digest') or response.get('digest')
            if not reported and not getattr(self, '_encoder_metadata_verified', False):
                raise ValueError('Custom embedding providers must report a resolved model digest')
            if reported is not None and reported != self.encoder_spec.model_digest:
                raise ValueError('Provider model digest does not match EncoderSpec')
        vector = self._validate_vector(vector)
        actual = response.get('model') if isinstance(response, dict) else getattr(response, 'model', None)
        actual = actual if isinstance(actual, str) and actual else self.model
        if cache:
            with self.connection:
                self.connection.execute('INSERT OR REPLACE INTO vectors VALUES (?, ?, ?)', (key, _json(vector), actual))
        return vector, actual, False

    def _views(self, record):
        for kind in self.representations:
            if kind == 'source' and (record.content_format == 'json' or record.type in {'table_cell', 'table_row_group'}):
                continue
            text = RepresentationSpec(kind).render(record)
            if not text.strip(): continue
            if self.encoder_spec and token_count(self.encoder_spec.document_template.replace('{text}', text)) > self.encoder_spec.max_input_tokens:
                if self.encoder_spec.overflow_policy != 'split':
                    self.encoder_spec.render(text)  # Raises a useful budget error before any publication.
                budget = self.encoder_spec.max_input_tokens - token_count(self.encoder_spec.document_template.replace('{text}', ''))
                if budget < 4: raise ValueError('Encoder template leaves no source-view budget')
                parts, current = [], ''
                # Preserve exact characters; prefer source line boundaries, split a long line only when necessary.
                for line in text.splitlines(keepends=True):
                    if token_count(current + line) <= budget:
                        current += line
                        continue
                    if current: parts.append(current); current = ''
                    for char in line:
                        if token_count(current + char) > budget:
                            parts.append(current); current = ''
                        current += char
                if current: parts.append(current)
            else: parts = [text]
            for part, value in enumerate(parts): yield kind, part, value

    async def _prepare_embeddings(self, views):
        unique = dict((text, label) for _, _, _, text, label in views)
        pending, prepared = iter(unique.items()), {}
        async def worker():
            for text, label in pending:
                _, _, reused = await self._embedding(text, label)
                prepared[text] = reused
        tasks = [asyncio.create_task(worker()) for _ in range(min(self.embedding_concurrency, len(unique)))]
        try: await asyncio.gather(*tasks)
        except BaseException:
            for task in tasks: task.cancel()
            await asyncio.gather(*tasks, return_exceptions=True)
            raise
        return prepared

    def _active_generation(self, document_id):
        row = self.connection.execute('SELECT generation_id FROM active_documents WHERE document_id=?', (document_id,)).fetchone()
        return row[0] if row else None

    def is_complete(self, document_id, *, generation=None):
        generation = generation or self._active_generation(document_id)
        if generation is None: return False
        row = self.connection.execute('SELECT embedded_count FROM generations WHERE id=? AND document_id=?', (generation, document_id)).fetchone()
        if row is None: return False
        expected = [r[0] for r in self.connection.execute('SELECT vector_id FROM object_representations WHERE generation_id=?', (generation,))]
        if not expected:
            expected = [r[0] for r in self.connection.execute('SELECT vector_id FROM objects WHERE generation_id=? AND vector_id IS NOT NULL', (generation,))]
        if len(expected) != row[0]: return False
        for start in range(0, len(expected), self.write_batch_size):
            batch = expected[start:start+self.write_batch_size]
            if set(self.collection.get(ids=batch, include=[])['ids']) != set(batch): return False
        return True

    async def index_file(self, semantics_file: str | Path) -> IndexingResult:
        """Publish a complete document, reusing cached vectors on rerun or retry.

        Exceptions (including cancellation) propagate; the old active generation
        remains accessible. Successfully computed vectors are cached for retry.
        """
        async with self._writer:
            source = Path(semantics_file).resolve()
            raw = source.read_bytes().decode('utf-8')
            records = collect_object_records(json.loads(raw), source, raw_json=raw, include_semantic_nodes=self.include_semantic_nodes)
            graph = self._document_graph(records, json.loads(raw), source)
            document_id = records[0].id
            signature = _digest([self.fingerprint, graph.revision.id, [asdict(record) for record in records]])
            previous = self._active_generation(document_id)
            if previous:
                row = self.connection.execute('SELECT signature FROM generations WHERE id=?', (previous,)).fetchone()
                if row['signature'] == signature and self.is_complete(document_id, generation=previous):
                    count = sum(1 for record in records for _ in self._views(record))
                    return IndexingResult(str(source), 'unchanged', document_id, previous,
                        len(records) - 1, 0, count, len(records) - count)
            generation = uuid4().hex
            preparation = self.catalog_path.parent / 'preparations' / (generation + '.json')
            atomic_json(preparation, {'generations': [generation], 'document_id': document_id, 'previous': previous, 'signature': signature})
            embedded = reused = skipped = 0
            views = [(record, kind, part, text, f'{source} at {record.object_path}')
                     for record in records for kind, part, text in self._views(record)]
            prepared = await self._prepare_embeddings(views) if self.embedding_concurrency > 1 else None
            used_texts, vector_ids, representation_rows = set(), {}, []
            payload = dict(ids=[], embeddings=[], documents=[], metadatas=[])
            for record, kind, part, text, label in views:
                vector, actual, was_reused = await self._embedding(text, label)
                if prepared is not None: was_reused = prepared[text] or text in used_texts
                used_texts.add(text)
                reused += int(was_reused); embedded += int(not was_reused)
                vector_id = _digest([generation, record.id]) if kind == 'summary' and part == 0 else _digest([generation, record.id, kind, part])
                if kind == 'summary' and part == 0: vector_ids[record.id] = vector_id
                rendered = self.encoder_spec.render(text) if self.encoder_spec else text
                representation_rows.append((generation, record.id, kind, part, vector_id,
                    hashlib.sha256(rendered.encode()).hexdigest(), _json({'object_id': record.id, 'source_locator': record.object_path, 'part': part,
                        'context_fields': ['section_headings', 'caption', 'headers'] if kind == 'source' else [],
                        'context_provenance': 'derived-context' if kind == 'source' else 'summary', 'revision_id': graph.revision.id})))
                payload['ids'].append(vector_id); payload['embeddings'].append(vector); payload['documents'].append(rendered)
                payload['metadatas'].append({'object_id': record.id, 'document_id': document_id,
                    'generation_id': generation, 'object_type': record.type, 'representation': kind,
                    'object_path': record.object_path, 'semantics_file': str(source),
                    'source_file': record.source_file, 'embed_model': self.model,
                    'actual_model': actual, 'config_fingerprint': self.fingerprint})
                if len(payload['ids']) >= self.write_batch_size:
                    self.collection.upsert(**payload)
                    payload = dict(ids=[], embeddings=[], documents=[], metadatas=[])
            if payload['ids']: self.collection.upsert(**payload)
            skipped = sum(not any(True for _ in self._views(record)) for record in records)
            catalog_rows = [(generation, record.id, _json(asdict(record)), vector_ids.get(record.id)) for record in records]
            # Guard against publishing a stale read if an unsupported second writer
            # has published this document while provider requests were running.
            with index_lock(self.catalog_path.parent):
                try:
                    self.connection.execute('BEGIN IMMEDIATE')
                    if self._active_generation(document_id) != previous:
                        raise RuntimeError('Another writer published this document; retry with one indexing writer')
                    self.connection.execute('INSERT INTO generations VALUES (?, ?, ?, ?, ?)',
                        (generation, document_id, str(source), signature, embedded + reused))
                    self.connection.executemany('INSERT INTO objects VALUES (?, ?, ?, ?)', catalog_rows)
                    self.connection.executemany('INSERT INTO object_representations VALUES (?,?,?,?,?,?,?)', representation_rows)
                    self.connection.execute('INSERT INTO document_graphs VALUES (?,?,?,?)', (generation, graph.document.id, graph.revision.id, _json(graph.to_dict())))
                    for record in records: self._insert_metadata(generation, asdict(record), vector_ids.get(record.id))
                    import time
                    self.connection.execute('INSERT INTO generation_lifecycle VALUES (?,?)', (generation, time.time()))
                    self.connection.execute('INSERT OR REPLACE INTO active_documents VALUES (?, ?)', (document_id, generation))
                    self.connection.commit()
                except BaseException:
                    self.connection.rollback()
                    raise
            preparation.unlink(missing_ok=True)
            try:
                self._write_manifest()
            except OSError as error:
                # The SQL pointer is authoritative; manifest repair must not repeat model work.
                self.manifest_error = str(error)
            return IndexingResult(str(source), 'indexed', document_id, generation,
                len(records) - 1, embedded, reused, skipped)

    def synchronize_manifest(self, manifest, *, dry_run=True):
        """Apply only explicit tombstones from a complete authoritative corpus manifest."""
        required = {'corpus_id', 'document_ids', 'tombstones', 'authoritative', 'complete'}
        if not required <= manifest.keys() or not manifest['authoritative'] or not manifest['complete']:
            raise ValueError('Synchronization requires a complete authoritative corpus manifest')
        if set(manifest['document_ids']) & set(manifest['tombstones']):
            raise ValueError('An active document cannot also be tombstoned')
        active = dict(self.connection.execute('SELECT document_id,generation_id FROM active_documents'))
        unknown = set(manifest['tombstones']) - active.keys()
        if unknown: raise ValueError(f'Unknown tombstone document IDs: {sorted(unknown)}')
        removed = {identifier: active[identifier] for identifier in manifest['tombstones']}
        if not dry_run:
            with index_lock(self.catalog_path.parent), self.connection:
                for identifier in removed:
                    source = self.connection.execute('SELECT source FROM generations WHERE id=?', (removed[identifier],)).fetchone()[0]
                    self.connection.execute('DELETE FROM active_documents WHERE document_id=? AND generation_id=?', (identifier, removed[identifier]))
                    self.connection.execute('INSERT OR REPLACE INTO corpus_tombstones VALUES (?,?,?)', (identifier, source, _digest(manifest)))
            self._write_manifest()
        return dict(dry_run=dry_run, tombstoned=removed)

    def cleanup_generations(self, *, dry_run=True, grace_seconds=86400, keep_previous=1, now=None):
        """Retain active, reader/preparation-pinned, recent and rollback generations."""
        import time
        if grace_seconds < 0 or keep_previous < 0: raise ValueError('Retention values must be nonnegative')
        now = time.time() if now is None else now
        with index_lock(self.catalog_path.parent):
            active = set(row[0] for row in self.connection.execute('SELECT generation_id FROM active_documents'))
            protected = active | protected_generations(self.catalog_path.parent)
            rows = self.connection.execute('''SELECT g.id,g.document_id,l.created FROM generations g
                LEFT JOIN generation_lifecycle l ON l.generation_id=g.id ORDER BY l.created DESC,g.id''').fetchall()
            retained = {}
            for generation, document, created in rows:
                if generation in active: continue
                if retained.get(document, 0) < keep_previous:
                    protected.add(generation); retained[document] = retained.get(document, 0)+1
                if created is None or now-created < grace_seconds: protected.add(generation)
            removable = [generation for generation, _, _ in rows if generation not in protected]
            if not dry_run:
                for generation in removable:
                    ids = [row[0] for row in self.connection.execute('SELECT vector_id FROM object_representations WHERE generation_id=?', (generation,))]
                    if not ids:
                        ids = [row[0] for row in self.connection.execute('SELECT vector_id FROM objects WHERE generation_id=? AND vector_id IS NOT NULL', (generation,))]
                    for start in range(0, len(ids), self.write_batch_size):
                        self.collection.delete(ids=ids[start:start+self.write_batch_size])
                    with self.connection:
                        for table in ('objects', 'object_metadata', 'object_representations', 'object_fts', 'object_terms', 'document_graphs', 'generation_lifecycle'):
                            self.connection.execute(f'DELETE FROM {table} WHERE generation_id=?', (generation,))
                        self.connection.execute('DELETE FROM generations WHERE id=?', (generation,))
            return dict(dry_run=dry_run, removable=removable, protected=sorted(protected))

    async def index_paths(self, paths) -> list[IndexingResult]:
        """Recursively discover *_semantics.json; report individual file failures.

        Explicit files must exist and match the suffix. Missing/invalid input roots
        fail visibly instead of reporting a misleading empty successful run.
        """
        if isinstance(paths, (str, Path)):
            paths = [paths]
        files = set()
        for value in paths:
            path = Path(value).resolve()
            if path.is_dir():
                files.update(path.rglob('*_semantics.json')); files.update(path.rglob('*_tasks.json'))
            elif path.is_file() and path.name.endswith(('_semantics.json', '_tasks.json')):
                files.add(path)
            else:
                raise ValueError(f'Expected a directory or existing *_semantics.json file: {path}')
        results = []
        for path in sorted(files):
            try:
                results.append(await self.index_file(path))
            except Exception as exc:
                results.append(IndexingResult(str(path), 'failed', error=f'{type(exc).__name__}: {exc}'))
        return results

    def _get_record(self, object_id, generation):
        row = self.connection.execute('SELECT record FROM objects WHERE id=? AND generation_id=?', (object_id, generation)).fetchone()
        return ObjectRecord(**json.loads(row[0])) if row else None

    def get_object(self, object_id: str) -> ObjectRecord | None:
        """Look up a logical ID in the currently active catalog, including unembedded objects."""
        row = self.connection.execute('''SELECT o.record FROM objects o
            JOIN active_documents a ON a.generation_id=o.generation_id WHERE o.id=?''', (object_id,)).fetchone()
        return ObjectRecord(**json.loads(row[0])) if row else None

    async def search(self, question: str, *, top_k: int = 5, document_ids=None, object_types=None) -> list[ObjectSearchHit]:
        """Cosine nearest summaries with raw evidence, ancestors and direct children.

        Filters accept sequences of IDs/types. Empty filter sequences match nothing.
        Distances are returned as distances, not probability or relevance scores.
        """
        if not isinstance(question, str) or not question.strip():
            raise ValueError('question must be a nonempty string')
        if isinstance(top_k, bool) or not isinstance(top_k, int) or top_k < 1:
            raise ValueError('top_k must be a positive integer')
        for name, values in [('document_ids', document_ids), ('object_types', object_types)]:
            if values is not None and (isinstance(values, str) or not all(isinstance(v, str) for v in values)):
                raise ValueError(f'{name} must be a sequence of strings')
            if values is not None and not values:
                return []
        with index_lock(self.catalog_path.parent):
            snapshot = self.connection.execute('''SELECT a.document_id, a.generation_id, g.embedded_count
                FROM active_documents a JOIN generations g ON g.id=a.generation_id''').fetchall()
            if document_ids is not None:
                snapshot = [row for row in snapshot if row['document_id'] in document_ids]
            snapshot = [row for row in snapshot if row['embedded_count']]
            if not snapshot:
                return []
            pin = pin_generations(self.catalog_path.parent, [row['generation_id'] for row in snapshot])
        try:
            vector, _, _ = await self._embedding(question, 'query', cache=False, query=True)
            candidates = []
            # Bound filter size while preserving the global top-k after merging.
            for start in range(0, len(snapshot), 100):
                batch = snapshot[start:start + 100]
                where = {'generation_id': {'$in': [row['generation_id'] for row in batch]}}
                if object_types is not None:
                    where = {'$and': [where, {'object_type': {'$in': list(object_types)}}]}
                count = sum(row['embedded_count'] for row in batch)
                limit = min(top_k, count)
                while True:
                    result = self.collection.query(query_embeddings=[vector], n_results=limit,
                        where=where, include=['distances', 'metadatas'])
                    unique = {}
                    for metadata, distance in zip(result['metadatas'][0], result['distances'][0]):
                        key = (metadata['object_id'], metadata['generation_id'])
                        unique[key] = min(unique.get(key, float('inf')), float(distance))
                    if len(unique) >= top_k or limit >= count or len(result['ids'][0]) < limit: break
                    limit = min(count, max(limit+1, limit*2))
                candidates.extend((distance, identifier, generation) for (identifier, generation), distance in unique.items())
            hits = []
            for distance, identifier, generation in sorted(candidates)[:top_k]:
                record = self._get_record(identifier, generation)
                if record is None:
                    raise RuntimeError(f'Active Chroma row has no catalog record: {identifier}')
                ancestors = []
                parent_id = record.parent_id
                while parent_id is not None:
                    parent = self._get_record(parent_id, generation)
                    if parent is None:
                        raise RuntimeError(f'Missing parent catalog record: {parent_id}')
                    ancestors.append(parent)
                    parent_id = parent.parent_id
                children = [self._get_record(child_id, generation) for child_id in record.children_ids]
                if any(child is None for child in children):
                    raise RuntimeError(f'Missing child catalog record for {identifier}')
                hits.append(ObjectSearchHit(distance, record, list(reversed(ancestors)), children))
            return hits
        finally:
            pin.unlink(missing_ok=True)


## Offline functional test

This end-to-end check uses an injected fake embedding provider, temporary SQLite,
and ephemeral Chroma. It verifies recursive indexing and ignored objects, source
preservation, unchanged reruns without model calls, retrieval with parent context,
and safe retry after a failed update. It never contacts Ollama or indexes a corpus.
Additional edge cases are covered in `tests/test_object_embedding.py`.

In [ ]:
# | hide
from tempfile import TemporaryDirectory
import chromadb


async def test_object_embedding_roundtrip():
    class OfflineProvider:
        def __init__(self):
            self.calls = []
            self.fail = False

        async def embed(self, model, input):
            await asyncio.sleep(0)
            if self.fail:
                raise RuntimeError('Simulated embedding failure')
            self.calls.append(input)
            return {'embeddings': [[float(len(input)), 1.0, 2.0]], 'model': model}

    tree = {'summary': 'Document', 'objects': [
        {'id': 1, 'type': 'list', 'content': '- raw item\n', 'summary': 'List', 'children': [
            {'id': '1.item.1', 'type': 'list_item', 'content': '- raw item\n', 'summary': 'Needle'},
            {'id': '1.figure.1', 'type': 'figure', 'content': '![](ignored.png)',
             'summary': '', 'summary_ignored': True}]}]}
    with TemporaryDirectory() as directory:
        source = Path(directory) / 'manual_semantics.json'
        source.write_text(json.dumps(tree), encoding='utf-8')
        original = source.read_bytes()
        database = chromadb.EphemeralClient()
        collection = database.create_collection('offline-object-test-' + uuid4().hex,
            metadata={'hnsw:space': 'cosine'}, embedding_function=None)
        provider = OfflineProvider()
        try:
            with ObjectEmbeddingIndex(Path(directory) / 'catalog.sqlite3', collection=collection,
                    llm_client=provider, model='offline-model', embedding_concurrency=2) as index:
                result = await index.index_file(source)
                assert (result.objects, result.embedded, result.skipped) == (3, 3, 1)
                assert source.read_bytes() == original
                records = collect_object_records(tree, source)
                ignored = next(record for record in records if record.type == 'figure')
                assert index.get_object(ignored.id).summary == ''
                requests = len(provider.calls)
                assert (await index.index_file(source)).status == 'unchanged'
                assert len(provider.calls) == requests
                hit, = await index.search('Needle', top_k=1, object_types=['list_item'])
                assert hit.record.content == '- raw item\n'
                assert [parent.type for parent in hit.ancestors] == ['document', 'list']
                assert hit.record.parent_id == hit.ancestors[-1].id
                tree['objects'][0]['children'][0]['summary'] = 'Updated needle'
                source.write_text(json.dumps(tree), encoding='utf-8')
                changed = source.read_bytes()
                provider.fail = True
                try:
                    await index.index_file(source)
                except RuntimeError as error:
                    assert 'Simulated embedding failure' in str(error)
                else:
                    raise AssertionError('A provider failure must prevent publication')
                assert index.get_object(hit.record.id).summary == 'Needle'
                provider.fail = False
                retried = await index.index_file(source)
                assert (retried.embedded, retried.reused) == (1, 2)
                assert index.get_object(hit.record.id).summary == 'Updated needle'
                assert source.read_bytes() == changed
        finally:
            database.delete_collection(collection.name)


_ = await test_object_embedding_roundtrip()